# 結晶検出＋粒径推定 統合パイプライン（Jupyter版）
元の顕微鏡画像（1枚丸ごと）から、YOLO（SAHIでパッチ分割+重複除去）で結晶を検出し、
各結晶をResNetで円相当径(µm)推定する。`inference/`フォルダをカレントディレクトリとして
このnotebookを実行すること。

> VSCodeで開く場合: 右上でカーネル（Python環境）を選択してから、上から順にセルを実行してください。

## 準備

In [ ]:
%matplotlib inline
from pathlib import Path

import pandas as pd
import torch
from PIL import Image, ImageDraw
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
import matplotlib.pyplot as plt

from resnet_utils import build_model, make_resnet_transform

IMAGE_PATH = "path/to/raw_image.png"   # ← 実際の画像パスに変更
YOLO_WEIGHTS = "../yolo_project/runs/detect/crystal_yolo/weights/best.pt"
RESNET_WEIGHTS = "../vscode_project/outputs/best_model.pth"

PATCH_SIZE = 640
OVERLAP_RATIO = 0.15
CONF = 0.25
OUTPUT_DIR = "results"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"デバイス: {device}")

assert Path(IMAGE_PATH).exists(), f"{IMAGE_PATH} が見つかりません。IMAGE_PATHを実際の画像パスに変更してください。"
assert Path(YOLO_WEIGHTS).exists(), f"{YOLO_WEIGHTS} が見つかりません。"
assert Path(RESNET_WEIGHTS).exists(), f"{RESNET_WEIGHTS} が見つかりません。"
print("準備完了")

## Step 1: YOLO検出（SAHIでパッチ分割 + 重複除去）

In [ ]:
detection_model = AutoDetectionModel.from_pretrained(
    model_type="ultralytics",
    model_path=YOLO_WEIGHTS,
    confidence_threshold=CONF,
    device=str(device),
)

sliced_result = get_sliced_prediction(
    IMAGE_PATH,
    detection_model,
    slice_height=PATCH_SIZE,
    slice_width=PATCH_SIZE,
    overlap_height_ratio=OVERLAP_RATIO,
    overlap_width_ratio=OVERLAP_RATIO,
)

orig_img = Image.open(IMAGE_PATH).convert("RGB")
W, H = orig_img.size

boxes = []
for pred in sliced_result.object_prediction_list:
    x1 = max(0, int(pred.bbox.minx))
    y1 = max(0, int(pred.bbox.miny))
    x2 = min(W, int(pred.bbox.maxx))
    y2 = min(H, int(pred.bbox.maxy))
    if x2 <= x1 or y2 <= y1:
        continue
    boxes.append((x1, y1, x2, y2, float(pred.score.value)))

print(f"検出数: {len(boxes)}件")

## Step 2: 各結晶をcropしてResNetで円相当径を推定

In [ ]:
resnet = build_model().to(device)
resnet.load_state_dict(torch.load(RESNET_WEIGHTS, map_location=device))
resnet.eval()
transform = make_resnet_transform()

records = []
with torch.no_grad():
    for (x1, y1, x2, y2, conf) in boxes:
        crop = orig_img.crop((x1, y1, x2, y2))
        x = transform(crop).unsqueeze(0).to(device)
        pred_um = resnet(x).item()
        records.append({
            "x1": x1, "y1": y1, "x2": x2, "y2": y2,
            "yolo_conf": conf, "diameter_um": pred_um,
        })

df = pd.DataFrame(records)

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / f"{Path(IMAGE_PATH).stem}_predictions.csv"
df.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"保存: {csv_path}")

if len(df):
    print(f"検出数: {len(df)}件  "
          f"平均粒径: {df['diameter_um'].mean():.2f}µm  "
          f"中央値: {df['diameter_um'].median():.2f}µm")
else:
    print("結晶が検出されませんでした。")

## Step 3: 可視化（検出枠 + 推定粒径）

In [ ]:
vis = orig_img.copy()
draw = ImageDraw.Draw(vis)
for _, row in df.iterrows():
    draw.rectangle((row.x1, row.y1, row.x2, row.y2), outline=(255, 0, 0), width=2)
    draw.text((row.x1, max(0, row.y1 - 12)), f"{row.diameter_um:.1f}", fill=(255, 0, 0))

vis_path = out_dir / f"{Path(IMAGE_PATH).stem}_annotated.png"
vis.save(vis_path)
print(f"可視化画像: {vis_path}")

plt.figure(figsize=(10, 10 * vis.height / vis.width))
plt.imshow(vis)
plt.axis("off")
plt.title(f"検出数: {len(df)}件")
plt.show()

## Step 4: 粒径分布のヒストグラム

In [ ]:
if len(df):
    plt.figure(figsize=(8, 5))
    plt.hist(df["diameter_um"], bins=30)
    plt.xlabel("推定粒径 (µm)")
    plt.ylabel("件数")
    plt.title(f"粒径分布（n={len(df)}）")
    plt.show()
else:
    print("結晶が検出されなかったため、ヒストグラムは表示できません。")